# Data Quickstart

Quick start checks for dataset readiness.

Steps:
- Verify key dataset folders.
- Review training data audit summary.
- Inspect recent experiment outputs.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

summary = {
    'folders': {},
    'reports': {},
    'experiments': {},
}

folders = [
    REPO_ROOT / 'data' / 'raw',
    REPO_ROOT / 'data' / 'processed',
    REPO_ROOT / 'data' / 'docs',
    REPO_ROOT / 'data' / 'dsa_docs',
    REPO_ROOT / 'models',
    REPO_ROOT / 'experiments',
]

for folder in folders:
    if not folder.exists():
        print('Missing:', folder)
        summary['folders'][str(folder)] = {'exists': False}
        continue
    items = list(folder.iterdir())
    summary['folders'][str(folder)] = {'exists': True, 'items': len(items)}
    print(folder.relative_to(REPO_ROOT), 'items:', len(items))


In [ ]:
# Review training data audit summary.
training_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if training_path.exists():
    data = json.loads(training_path.read_text(encoding='utf-8'))
    required = data.get('required', [])
    optional = data.get('optional', [])
    missing_required = [item for item in required if item.get('status') != 'ok']
    summary['reports']['missing_required'] = len(missing_required)
    print('Required datasets:', len(required))
    print('Missing required:', len(missing_required))
    for item in missing_required[:10]:
        print(' -', item.get('name'))
else:
    print('Missing TRAINING_DATA.json')


In [ ]:
# Inspect recent experiment outputs.
report_path = REPO_ROOT / 'experiments' / 'report_summary.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    summary['experiments']['report_summary'] = report
    print('Experiment report summary keys:', list(report.keys()))
else:
    print('Missing experiments/report_summary.json')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_data_quickstart_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize data-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'data' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No data entries found in TRAINING_DATA.json')
    else:
        print('data datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
